In [1]:
# ── Install (run once, then restart kernel) ───────────────────────────────────
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn openpyxl -q
print('Libraries installed ✓')

Libraries installed ✓


In [2]:
import sys
sys.path.append('../../shared')

import pandas as pd
import numpy as np

# ── Load raw CSV ──────────────────────────────────────────────────────────────
RAW_PATH = '../../data/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(RAW_PATH)

print(f'Raw shape : {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

Raw shape : 7,043 rows × 21 columns


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [3]:
# ── 1. Data types & missing values ───────────────────────────────────────────
print('=== dtypes ===')
print(df.dtypes)
print('\n=== Missing values ===')
print(df.isnull().sum()[df.isnull().sum() > 0])

=== dtypes ===
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

=== Missing values ===
Series([], dtype: int64)


In [4]:
# ── 2. Fix TotalCharges (stored as string) ────────────────────────────────────
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# New customers (tenure=0) have blank TotalCharges → fill with 0
print(f'NaN in TotalCharges: {df["TotalCharges"].isna().sum()} rows (tenure=0 customers)')
df['TotalCharges'].fillna(0, inplace=True)

NaN in TotalCharges: 11 rows (tenure=0 customers)


C:\Users\USER\AppData\Local\Temp\ipykernel_6876\2360755861.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(0, inplace=True)


In [5]:
# ── 3. Feature engineering ────────────────────────────────────────────────────

# Binary churn flag
df['Churn_Binary'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Readable senior citizen label
df['SeniorCitizen_Label'] = df['SeniorCitizen'].map({1: 'Senior', 0: 'Non-Senior'})

# Tenure buckets
df['Tenure_Group'] = pd.cut(
    df['tenure'],
    bins=[0, 12, 24, 48, 72],
    labels=['0-12 mo', '13-24 mo', '25-48 mo', '49-72 mo'],
    include_lowest=True
)

# Monthly charge tier
df['Charge_Tier'] = pd.cut(
    df['MonthlyCharges'],
    bins=[0, 35, 65, 95, 120],
    labels=['Low (<$35)', 'Mid ($35-65)', 'High ($65-95)', 'Premium (>$95)']
)

print('Feature engineering complete ✓')
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df[['tenure', 'Tenure_Group', 'MonthlyCharges', 'Charge_Tier', 'Churn', 'Churn_Binary']].head(5)

Feature engineering complete ✓
Final shape: 7,043 rows × 25 columns


,tenure,Tenure_Group,MonthlyCharges,Charge_Tier,Churn,Churn_Binary
0,1,0-12 mo,29.85,Low (<$35),No,0
1,34,25-48 mo,56.95,Mid ($35-65),No,0
2,2,0-12 mo,53.85,Mid ($35-65),Yes,1
3,45,25-48 mo,42.30,Mid ($35-65),No,0
4,2,0-12 mo,70.70,High ($65-95),Yes,1


In [6]:
# ── 4. Sanity checks ──────────────────────────────────────────────────────────
assert df['Churn_Binary'].isna().sum() == 0, 'Churn_Binary has nulls'
assert df['TotalCharges'].isna().sum() == 0, 'TotalCharges has nulls'
assert df['tenure'].min() >= 0, 'Negative tenure found'
print('All sanity checks passed ✓')

print(f'\nChurn rate: {df["Churn_Binary"].mean()*100:.2f}%')
print(f'Tenure range: {df["tenure"].min()} – {df["tenure"].max()} months')
print(f'Monthly charges: ${df["MonthlyCharges"].min()} – ${df["MonthlyCharges"].max()}')

All sanity checks passed ✓

Churn rate: 26.54%
Tenure range: 0 – 72 months
Monthly charges: $18.25 – $118.75


In [9]:
# ── 5. Save cleaned dataset ───────────────────────────────────────────────────
import os
os.makedirs('../outputs', exist_ok=True)

df.to_csv('../outputs/telco_cleaned.csv', index=False)
print('Saved → eda-insights/outputs/telco_cleaned.csv ✓')
print('\nNext → open 02_eda_charts.ipynb')

Saved → eda-insights/outputs/telco_cleaned.csv ✓

Next → open 02_eda_charts.ipynb


In [ ]:
# ── Chart 1: Overall churn split ──────────────────────────────────────────────
total    = len(df)
churned  = df['Churn_Binary'].sum()
retained = total - churned

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Pie
axes[0].pie(
    [churned, retained],
    labels=['Churned', 'Retained'],
    colors=[CHURN_COLOR, RETAIN_COLOR],
    autopct='%1.1f%%',
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[0].set_title('Overall Churn Split')

# Bar
bars = axes[1].bar(
    ['Retained', 'Churned'], [retained, churned],
    color=[RETAIN_COLOR, CHURN_COLOR], width=0.5, edgecolor='white'
)
label_bars_v(axes[1], bars, fmt='{:,.0f}', pad=40)
axes[1].set_title('Customer Count by Churn Status')
axes[1].set_ylabel('Number of customers')
axes[1].set_ylim(0, retained * 1.15)

plt.tight_layout()
plt.savefig('../outputs/chart_01_churn_overview.png', bbox_inches='tight')
plt.show()
print('Chart 01 saved ✓')